In [1]:
import datetime as dt
import glob
import pandas as pd

# 1. Đọc dữ liệu (C01)
search_path = r'D:\KhaiThacDuLieu\**\*online_retail*.csv'
found_files = glob.glob(search_path, recursive=True)
file_path = found_files[0]
df = pd.read_csv(file_path, encoding='ISO-8859-1')

# 2. Làm sạch dữ liệu (C03)
df_clean = df.dropna(subset=['Customer ID']).copy()
df_clean['Invoice'] = df_clean['Invoice'].astype(str)
df_clean = df_clean[~df_clean['Invoice'].str.startswith('C')]
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['TotalSum'] = df_clean['Quantity'] * df_clean['Price']
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int).astype(str)

# Xuất file sạch C03
df_clean.to_parquet(
    'D:\\KhaiThacDuLieu\\online_retail_clean.parquet', index=False
)
print('✅ ĐÃ XUẤT FILE online_retail_clean.parquet THÀNH CÔNG!')

# 3. Tính toán RFM & Đặc trưng Loyalty (C02 & C04)
snapshot_date = df_clean['InvoiceDate'].max() + dt.timedelta(days=1)

rfm = (
    df_clean.groupby('Customer ID')
    .agg({
        'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
        'Invoice': 'nunique',  # Frequency
        'TotalSum': 'sum',  # Monetary
        'Quantity': 'sum',
    })
    .reset_index()
)

rfm.columns = [
    'Customer_ID',
    'Recency',
    'Frequency',
    'Monetary',
    'Total_Quantity',
]
rfm['AOV'] = rfm['Monetary'] / rfm['Frequency']

# 4. Phân khúc RFM (C06)
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]
)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1, 2, 3, 4])
rfm['RFM_Score'] = (
    rfm['R_Score'].astype(str)
    + rfm['F_Score'].astype(str)
    + rfm['M_Score'].astype(str)
)


def segment_customer(row):
  if row['RFM_Score'] in ['444', '443', '344', '434']:
    return 'Champions (VIP)'
  elif row['R_Score'] in [3, 4] and row['F_Score'] in [3, 4]:
    return 'Loyal Customers'
  elif row['R_Score'] in [3, 4] and row['F_Score'] in [1, 2]:
    return 'Promising / New'
  elif row['R_Score'] in [1, 2] and row['F_Score'] in [3, 4]:
    return 'At Risk'
  else:
    return 'Hibernating / Lost'


rfm['Segment'] = rfm.apply(segment_customer, axis=1)

# Xuất file kết quả C04
rfm.to_csv('D:\\KhaiThacDuLieu\\features_loyalty_retail.csv', index=False)
print('✅ ĐÃ XUẤT FILE features_loyalty_retail.csv THÀNH CÔNG!')

print('\n=== SỐ LƯỢNG KHÁCH HÀNG THEO PHÂN KHÚC ===')
print(rfm['Segment'].value_counts())

✅ ĐÃ XUẤT FILE online_retail_clean.parquet THÀNH CÔNG!
✅ ĐÃ XUẤT FILE features_loyalty_retail.csv THÀNH CÔNG!

=== SỐ LƯỢNG KHÁCH HÀNG THEO PHÂN KHÚC ===
Segment
Hibernating / Lost    2045
Champions (VIP)       1182
Promising / New        894
At Risk                880
Loyal Customers        877
Name: count, dtype: int64
